In [1]:
# Figuras de la etapa experimental: selección de tubo y kit + estabilidad
# =====================================================================
# Argumento que muestran las figuras:
#   Fig. 1 -> Quick-RNA Miniprep Plus es el kit que da ARN de buena calidad
#             (RNeasy, Dynabeads y PAXgene+Quick fallan). Con Quick, Shield y
#             EDTA rinden parecido.
#   Fig. 2 -> Se elige Shield porque estabiliza el ARN durante el transporte.
#   Fig. 3 -> El ARN de sangre total es válido frente al gold standard (PBMCs).
#
# Uso:       python figuras_tesis_experimental.py
# Necesita:  pip install pandas matplotlib numpy

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# =====================================================================
# PARÁMETROS
# =====================================================================
ENTRADA = "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/data/experimental_fondef/Planilla_Fondef_ID24I10045 - INFO_GENERAL _RNA (2).tsv"
SALIDA = "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/figuras_experimental"

# Excluir filas con Consentimiento = "No"  (confírmalo con tu tutor/a)
EXCLUIR_SIN_CONSENTIMIENTO = False

# Corregir el kit de muestras mal anotadas en la planilla, si corresponde.
# Ejemplo: si las 0010 en realidad fueron RNeasy, descomenta la línea:
CORRECCIONES_KIT = {
    # r"I\+D_LLC_0010": "RNeasy",
}

# Gold standard PBMC (Fig. 3): PBMCs frescas (0 h) extraídas con estos kits
KITS_PBMC = ["RNeasy"]
EXCLUIR_PBMC = [
    "Ensayo RNAlater",        # ensayo aparte de conservación en RNAlater
    r"I\+D_LLC_0015",         # PBMC congelada en 2023 (otro proyecto, sin consentimiento)
]

# Muestras excluidas por una causa TÉCNICA documentada (no por su resultado).
# Cada exclusión se aplica en todas las figuras y se imprime al correr el script.
# Descomenta solo las que puedas justificar en Métodos:
EXCLUIR_MUESTRAS = {
    # r"I\+D_LLC_0010": "kit mal anotado / primer intento, ver cuaderno 07/11/2024",
    # r"I\+D_LLC_0006$": "primera extracción de PBMC (prueba piloto)",
}

# Criterios de calidad
UMBRAL_CONC = 20          # ng/µl mínimo (ajústalo al input que pide tu kit de bibliotecas)
UMBRAL_280 = (1.8, 2.2)
UMBRAL_230 = 1.7
UMBRAL_RIN = 7.0

# Colores
MORADO = "#6250d6"    # PBMC
NARANJO = "#eb6834"   # Shield + Quick (seleccionada)
AZUL = "#2a78d6"      # EDTA + Quick
GRIS = "#a9a8a3"      # otras combinaciones
FONDO_QUICK = "#f4f3f0"
FONDO_ELEGIDA = "#fdeee6"
VERDE_OK = "#e9f3ea"


# =====================================================================
# 1. LEER Y LIMPIAR
# =====================================================================
df = pd.read_csv(ENTRADA, sep="\t", header=2, dtype=str)  # títulos en la 3ª fila
df.columns = df.columns.str.strip()
df = df.rename(columns={"Concentración Nanodrop (ng/ul)": "conc"})
df["codigo"] = df["CODIGO DE MUESTRA"].fillna("").str.strip()

for patron, motivo in EXCLUIR_MUESTRAS.items():
    sacar = df["codigo"].str.contains(patron)
    print(f"Excluidas {sacar.sum()} filas ({patron}): {motivo}")
    df = df[~sacar]

# Guardar el texto de RIN antes de convertir ("-" = medido pero sin RIN calculable)
df["RIN_texto"] = df["RIN"].fillna("").str.strip()

for col in ["conc", "260/280", "260/230", "RIN"]:
    df[col] = pd.to_numeric(df[col].str.strip().str.replace(",", "."), errors="coerce")
df.loc[df["RIN"] <= 0, "RIN"] = np.nan

# Tubo y kit con nombres cortos
df["tubo"] = df["TIPO DE TUBO"].str.strip().replace({"SHIELD": "Shield", "PAXgene": "PAXgene"})
k = df["KIT"].str.lower().fillna("")
df["kit"] = np.select([k.str.contains("quick"), k.str.contains("rneasy"),
                       k.str.contains("dyna"), k.str.contains("pure")],
                      ["Quick", "RNeasy", "Dynabeads", "PureLink"], default="")
for patron, kit_ok in CORRECCIONES_KIT.items():
    df.loc[df["codigo"].str.contains(patron), "kit"] = kit_ok

# Tiempo en horas ("0hrs", "24 hrs", "1 mes", "2 meses"...)
num = df["Tiempo"].str.extract(r"(\d+)")[0].astype(float)
df["horas"] = np.where(df["Tiempo"].str.contains("mes", case=False, na=False), num * 720, num)

# Temperatura
df["temp"] = df["grados C"].str.strip().replace({"4 C": "4", "TA": "TA"})

# Filtros: sangre total, sin blancos (código termina en "B")
es_sangre = df["TIPO DE MUESTRA"].str.contains("sangre total", case=False, na=False)
es_blanco = df["codigo"].str.contains(r"R\d+B$")
sangre = df[es_sangre & ~es_blanco]
if EXCLUIR_SIN_CONSENTIMIENTO:
    sangre = sangre[sangre["Consentimiento"].fillna("").str.strip() != "No"]

base = sangre[sangre.horas <= 2]                       # sin estrés de transporte
shield_quick = sangre[(sangre.tubo == "Shield") & (sangre.kit == "Quick")]

# PBMC gold standard (para Fig. 3)
es_pbmc = df["TIPO DE MUESTRA"].str.contains("pbmc", case=False, na=False)
pbmc = df[es_pbmc & (df.horas == 0) & df.kit.isin(KITS_PBMC)]
for patron in EXCLUIR_PBMC:
    pbmc = pbmc[~pbmc["codigo"].str.contains(patron)]
if EXCLUIR_SIN_CONSENTIMIENTO:
    pbmc = pbmc[pbmc["Consentimiento"].fillna("").str.strip() != "No"]


# =====================================================================
# Ayudas de formato
# =====================================================================
plt.rcParams.update({"font.size": 8.5, "axes.spines.top": False, "axes.spines.right": False})
coma = lambda v, p: f"{v:g}".replace(".", ",")


def estilo(ax, letra):
    ax.yaxis.set_major_formatter(coma)
    ax.yaxis.grid(True, color="#e6e5e1", lw=0.6)
    ax.set_axisbelow(True)
    ax.text(-0.24, 1.06, letra, transform=ax.transAxes, fontsize=12, fontweight="bold")


def puntos(ax, x, y, color, mini=None, maxi=None, s=16, marker="o", zorder=3):
    """Dibuja puntos: rellenos si cumplen el criterio [mini, maxi],
    vacíos (solo borde) si quedan fuera del rango aceptable."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.ones(len(y), bool)
    if mini is not None:
        ok &= y >= mini
    if maxi is not None:
        ok &= y <= maxi
    ax.scatter(x[ok], y[ok], s=s, color=color, marker=marker, edgecolor="white", lw=0.5, zorder=zorder)
    ax.scatter(x[~ok], y[~ok], s=s, facecolor="white", edgecolor=color, marker=marker, lw=1.0, zorder=zorder)


def guardar(fig, nombre):
    os.makedirs(SALIDA, exist_ok=True)
    fig.savefig(os.path.join(SALIDA, nombre + ".png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(SALIDA, nombre + ".pdf"), bbox_inches="tight")
    plt.close(fig)
    print("Guardada:", nombre)


# =====================================================================
# FIGURA 1 – Selección de kit y tubo
# =====================================================================
# Orden: primero los kits descartados, luego el bloque Quick-RNA (con la elegida al final)
grupos = [("Shield", "RNeasy"), ("EDTA", "Dynabeads"),
          ("PAXgene", "Quick"), ("EDTA", "Quick"), ("Shield", "Quick")]
ELEGIDA = ("Shield", "Quick")
inicio_quick = 2  # índice donde empieza el bloque Quick-RNA


def color_de(t, k):
    if (t, k) == ELEGIDA:
        return NARANJO
    if (t, k) == ("EDTA", "Quick"):
        return AZUL
    return GRIS


paneles = [("conc", "Concentración NanoDrop (ng/µl)", UMBRAL_CONC, 1500, (0.25, 1500)),
           ("260/280", "Relación A260/280", *UMBRAL_280, (0.5, 3)),
           ("260/230", "Relación A260/230", UMBRAL_230, 2.6, (0, 2.6)),
           ("RIN", "RIN (TapeStation)", UMBRAL_RIN, 10.3, (0, 10.3))]

fig, axs = plt.subplots(2, 2, figsize=(7.4, 6.4))
np.random.seed(1)

for ax, (col, titulo, mini, maxi, ylim), letra in zip(axs.flat, paneles, "ABCD"):
    trans = ax.get_xaxis_transform()

    # Fondo del bloque Quick-RNA y de la combinación elegida
    ax.add_patch(plt.Rectangle((inicio_quick - 0.5, 0), len(grupos) - inicio_quick, 1,
                               transform=trans, color=FONDO_QUICK, zorder=0, lw=0))
    j = grupos.index(ELEGIDA)
    ax.add_patch(plt.Rectangle((j - 0.45, 0), 0.9, 1, transform=trans, facecolor=FONDO_ELEGIDA,
                               edgecolor=NARANJO, lw=1.4, zorder=0.5, clip_on=False))

    # Rango aceptable
    if mini is not None:
        ax.axhspan(mini, maxi, color=VERDE_OK, zorder=0.2, alpha=0.7)
        ax.axhline(mini, color="green", ls="--", lw=0.9, zorder=1)
        if col == "260/280":
            ax.axhline(maxi, color="green", ls="--", lw=0.9, zorder=1)

    for i, (t, kk) in enumerate(grupos):
        g = base[(base.tubo == t) & (base.kit == kk)]
        v = g[col].dropna()
        n = len(v)
        c = color_de(t, kk)

        # Concentración ≤ 0 (escala log): punto hueco en la base
        if col == "conc" and (v <= 0).any():
            n0 = int((v <= 0).sum())
            ax.scatter(i + np.random.uniform(-0.13, 0.13, n0), np.full(n0, 0.35), s=16,
                       facecolor="white", edgecolor=c, lw=0.9, zorder=3)
        # RIN no calculable ("-"): círculo vacío en la base (no cumple)
        if col == "RIN":
            n_sin = int(g["RIN_texto"].isin(["-", "0"]).sum())
            if n_sin:
                ax.scatter(i + np.random.uniform(-0.13, 0.13, n_sin), np.full(n_sin, 0.5), s=16,
                           facecolor="white", edgecolor=c, lw=1.0, zorder=3, clip_on=False)

        mediana = v.median() if n else None
        vp = v[v > 0] if col == "conc" else v
        maxi_ok = maxi if col == "260/280" else None   # solo 260/280 tiene máximo
        puntos(ax, i + np.random.uniform(-0.13, 0.13, len(vp)), vp, c, mini, maxi_ok,
               s=18 if c != GRIS else 13)
        if mediana is not None:
            ax.plot([i - 0.28, i + 0.28], [mediana] * 2, color="black", lw=1.6, zorder=4)
        ax.text(i, 1.01, f"n={n}", transform=trans, ha="center", fontsize=6.5, color="gray")

    # Ejes
    ax.set_xticks(range(len(grupos)))
    ax.set_xticklabels([f"{t}\n{kk}" for t, kk in grupos], fontsize=7)
    lbl = ax.get_xticklabels()[j]
    lbl.set_fontweight("bold"); lbl.set_color(NARANJO)
    ax.set_xlim(-0.6, len(grupos) - 0.4)
    ax.set_ylabel(titulo)
    if col == "conc":
        ax.set_yscale("log")
    ax.set_ylim(ylim)
    estilo(ax, letra)

    # Etiqueta del bloque Quick-RNA bajo los nombres
    ax.annotate("", xy=(inicio_quick - 0.4, -0.27), xytext=(len(grupos) - 0.6, -0.27),
                xycoords=trans, arrowprops=dict(arrowstyle="-", color="#52514e", lw=0.8))
    ax.text((inicio_quick + len(grupos) - 1) / 2, -0.29, "Kit Quick-RNA", transform=trans,
            ha="center", va="top", fontsize=6.8, style="italic", color="#52514e")

leyenda = [Patch(facecolor=FONDO_ELEGIDA, edgecolor=NARANJO, label="Shield + Quick-RNA (seleccionada)"),
           Line2D([], [], marker="o", ls="", color=AZUL, label="EDTA + Quick-RNA"),
           Line2D([], [], marker="o", ls="", color=GRIS, label="Otras combinaciones"),
           Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor="gray",
                  label="No cumple (fuera de rango, ≤0 o sin RIN)"),
           Line2D([], [], color="black", lw=1.6, label="Mediana"),
           Patch(color=VERDE_OK, label="Rango aceptable")]
fig.legend(handles=leyenda, loc="lower center", ncol=3, frameon=False, fontsize=7.2)
fig.suptitle("Selección de tubo de recolección y kit de extracción",
             x=0.02, ha="left", fontsize=10.5, fontweight="bold")
fig.tight_layout(rect=(0, 0.07, 1, 0.98), h_pad=2.5)
guardar(fig, "fig_5_2_seleccion_tubo_kit")


# =====================================================================
# FIGURA 2 – Estabilidad del ARN en tubo Shield (con Quick-RNA)
# =====================================================================
sq = shield_quick
rin0 = sq[sq.horas == 0]["RIN"].dropna()        # punto de partida: todas las muestras a 0 h
condiciones = [("TA", "Temperatura ambiente", NARANJO, "o"),
               ("-20", "−20 °C", AZUL, "s")]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(7.4, 3.1), gridspec_kw={"width_ratios": [2.2, 1]})

# --- Panel A: transporte (horas) ---
axA.axhspan(UMBRAL_RIN, 10.3, color=VERDE_OK, zorder=0)
axA.axhline(UMBRAL_RIN, color="green", ls="--", lw=0.9)
puntos(axA, np.random.uniform(-2, 2, len(rin0)), rin0, GRIS, UMBRAL_RIN, s=12)
axA.plot([-4, 4], [rin0.median()] * 2, color="black", lw=1.6, zorder=4)

for temp, nombre, c, m in condiciones:
    s = sq[(sq.temp == temp) & (sq.horas > 0) & (sq.horas <= 200) & sq.RIN.notna()]
    if s.empty:
        continue
    prom = s.groupby("horas")["RIN"].median()
    axA.plot([0] + list(prom.index), [rin0.median()] + list(prom.values), color=c, lw=2, zorder=2)
    puntos(axA, s.horas, s.RIN, c, UMBRAL_RIN, s=28, marker=m, zorder=5)
    dy = 0.7 if temp == "-20" else 0          # subir la etiqueta de −20 °C para no tapar puntos
    axA.text(prom.index[-1] + 5, prom.values[-1] + dy, nombre, color="black", fontsize=7.5, va="center")

axA.set_xticks([0, 24, 48, 72, 96, 168])
axA.set_xlim(-8, 215)
axA.set_ylim(0, 10.3)
axA.set_xlabel("Tiempo desde la toma (h)")
axA.set_ylabel("RIN")
axA.set_title("Transporte simulado", loc="left", fontsize=9, fontweight="bold", pad=14)
axA.text(3, 0.6, f"0 h: n={len(rin0)}", fontsize=6.5, color="gray")
estilo(axA, "A")

# --- Panel B: almacenamiento a −80 °C (meses) ---
largo = [("0 h", rin0),
         ("1 mes", sq[(sq.temp == "-80") & (sq.horas == 720)]["RIN"].dropna()),
         ("2 meses", sq[(sq.temp == "-80") & (sq.horas == 1440)]["RIN"].dropna())]
axB.axhspan(UMBRAL_RIN, 10.3, color=VERDE_OK, zorder=0)
axB.axhline(UMBRAL_RIN, color="green", ls="--", lw=0.9)
for i, (nombre, v) in enumerate(largo):
    c = GRIS if i == 0 else "#6250d6"
    puntos(axB, i + np.random.uniform(-0.12, 0.12, len(v)), v, c, UMBRAL_RIN, s=22 if i else 12)
    if len(v):
        axB.plot([i - 0.25, i + 0.25], [v.median()] * 2, color="black", lw=1.6, zorder=4)
    axB.text(i, 1.01, f"n={len(v)}", transform=axB.get_xaxis_transform(), ha="center",
             fontsize=6.5, color="gray")
axB.set_xticks(range(len(largo)))
axB.set_xticklabels([x[0] for x in largo], fontsize=7.5)
axB.set_xlim(-0.6, len(largo) - 0.4)
axB.set_ylim(0, 10.3)
axB.set_title("Almacenamiento a −80 °C", loc="left", fontsize=9, fontweight="bold", pad=14)
estilo(axB, "B")

fig.legend(handles=[Line2D([], [], marker="o", ls="", color=GRIS, label="RIN ≥ 7"),
                    Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor=GRIS,
                           label="RIN < 7 (no cumple)"),
                    Line2D([], [], color="black", lw=1.6, label="Mediana a 0 h / por grupo")],
           loc="lower center", ncol=3, frameon=False, fontsize=7.3)
fig.suptitle("Estabilidad del ARN en tubos Shield (Quick-RNA Miniprep Plus)",
             x=0.02, ha="left", fontsize=10.5, fontweight="bold")
fig.tight_layout(rect=(0, 0.07, 1, 0.95))
guardar(fig, "fig_5_3_estabilidad_shield")


# =====================================================================
# FIGURA 3 – PBMCs (gold standard) vs sangre total (Shield + Quick, 0 h)
# =====================================================================
sangre0 = shield_quick[shield_quick.horas == 0]
grupos3 = [("PBMC", pbmc, MORADO), ("Sangre\ntotal", sangre0, NARANJO)]
# (columna, título del panel, etiqueta eje y, mínimo, máximo, límites eje y)
paneles3 = [("conc", "Concentración", "Concentración (ng/µl)", UMBRAL_CONC, 1500, (0.25, 1500)),
            ("260/280", "Pureza: proteínas", "Relación A260/A280", *UMBRAL_280, (1.0, 3.0)),
            ("260/230", "Pureza: sales/reactivos", "Relación A260/A230", UMBRAL_230, 2.6, (0, 2.6)),
            ("RIN", "Integridad", "RIN", UMBRAL_RIN, 10.3, (0, 10.3))]

fig, axs = plt.subplots(1, 4, figsize=(7.8, 3.4))
print("\n--- PBMC vs sangre total (medianas) ---")

for ax, (col, titulo, eje_y, mini, maxi, ylim), letra in zip(axs, paneles3, "ABCD"):
    if mini is not None:
        ax.axhspan(mini, maxi, color=VERDE_OK, zorder=0)
        ax.axhline(mini, color="green", ls="--", lw=0.9)
        if col == "260/280":
            ax.axhline(maxi, color="green", ls="--", lw=0.9)

    for i, (nombre, g, c) in enumerate(grupos3):
        v = g[col].dropna()
        if col == "conc" and (v <= 0).any():          # lecturas ≤ 0: punto hueco en la base
            n0 = int((v <= 0).sum())
            ax.scatter(i + np.random.uniform(-0.15, 0.15, n0), np.full(n0, 0.35), s=16,
                       facecolor="white", edgecolor=c, lw=0.9, zorder=3)
        vp = v[v > 0] if col == "conc" else v
        maxi_ok = maxi if col == "260/280" else None
        puntos(ax, i + np.random.uniform(-0.15, 0.15, len(vp)), vp, c, mini, maxi_ok,
               s=22 if i == 0 else 14)
        if len(v):
            ax.plot([i - 0.3, i + 0.3], [v.median()] * 2, color="black", lw=1.6, zorder=4)
        ax.text(i, 1.01, f"n={len(v)}", transform=ax.get_xaxis_transform(),
                ha="center", fontsize=6.5, color="gray")

    a, b = pbmc[col].dropna(), sangre0[col].dropna()
    print(f"{col:8s} PBMC mediana {a.median():6.2f} (n={len(a)}) | "
          f"sangre mediana {b.median():6.2f} (n={len(b)})")

    # Etiquetas de ejes
    ax.set_xlabel("Tipo de muestra", fontsize=7.5)
    ax.set_ylabel(eje_y, fontsize=7.5, labelpad=2)

    ax.set_xticks([0, 1])
    ax.set_xticklabels([g[0] for g in grupos3], fontsize=7.5)
    ax.set_xlim(-0.6, 1.6)
    ax.set_title(titulo, fontsize=8.5, fontweight="bold", pad=14)
    if col == "conc":
        ax.set_yscale("log")
        ax.set_yticks([1, 10, 100, 1000])
    ax.set_ylim(ylim)
    ax.yaxis.set_major_formatter(coma)
    ax.yaxis.grid(True, color="#e6e5e1", lw=0.6)
    ax.set_axisbelow(True)
    ax.text(-0.42, 1.1, letra, transform=ax.transAxes, fontsize=12, fontweight="bold")

leyenda = [Line2D([], [], marker="o", ls="", color=MORADO, label="PBMCs aisladas (gold standard, RNeasy)"),
           Line2D([], [], marker="o", ls="", color=NARANJO, label="Sangre total (Shield + Quick-RNA)"),
           Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor="gray",
                  label="No cumple (fuera de rango o ≤0)"),
           Line2D([], [], color="black", lw=1.6, label="Mediana"),
           Patch(color=VERDE_OK, label="Rango aceptable")]
fig.legend(handles=leyenda, loc="lower center", ncol=3, frameon=False, fontsize=7.3)
fig.suptitle("Calidad del ARN: PBMCs aisladas vs. sangre total (0 h)",
             x=0.02, ha="left", fontsize=10.5, fontweight="bold")
fig.tight_layout(rect=(0, 0.1, 1, 0.96), w_pad=2.0)
guardar(fig, "fig_5_4_pbmc_vs_sangre")
print("Muestras PBMC usadas:", ", ".join(pbmc["codigo"]))


# =====================================================================
# Números para el texto de resultados
# =====================================================================
print("\n--- Resumen (sangre total, 0–2 h) ---")
for t, kk in grupos:
    g = base[(base.tubo == t) & (base.kit == kk)]
    r230 = g["260/230"].dropna(); rin = g["RIN"].dropna(); c = g["conc"].dropna()
    print(f"{t:8s} + {kk:9s} n={len(g):3d} | conc mediana {c.median():7.1f} ng/µl"
          f" | 260/280 mediana {g['260/280'].median():.2f}"
          f" | 260/230≥{UMBRAL_230}: {(r230 >= UMBRAL_230).sum()}/{len(r230)}"
          f" | RIN≥{UMBRAL_RIN}: {(rin >= UMBRAL_RIN).sum()}/{len(rin)}"
          f" (mediana {rin.median() if len(rin) else float('nan'):.1f})")

print("\n--- Estabilidad Shield + Quick (RIN mediana) ---")
print(sq[sq.RIN.notna()].groupby(["temp", "horas"])["RIN"].agg(["count", "median"]).to_string())

Guardada: fig_5_2_seleccion_tubo_kit
Guardada: fig_5_3_estabilidad_shield

--- PBMC vs sangre total (medianas) ---
conc     PBMC mediana  55.70 (n=5) | sangre mediana 122.40 (n=45)
260/280  PBMC mediana   2.05 (n=5) | sangre mediana   2.06 (n=45)
260/230  PBMC mediana   0.93 (n=5) | sangre mediana   1.81 (n=45)
RIN      PBMC mediana   9.40 (n=5) | sangre mediana   7.90 (n=37)
Guardada: fig_5_4_pbmc_vs_sangre
Muestras PBMC usadas: I+D_LLC_0006, I+D_LLC_0008, I+D_LLC_0008R1, I+D_LLC_0014, I+D_LLC_0014R1

--- Resumen (sangre total, 0–2 h) ---
Shield   + RNeasy    n= 27 | conc mediana     6.7 ng/µl | 260/280 mediana 2.06 | 260/230≥1.7: 0/26 | RIN≥7.0: 3/6 (mediana 7.0)
EDTA     + Dynabeads n=  2 | conc mediana     9.2 ng/µl | 260/280 mediana 1.71 | 260/230≥1.7: 0/2 | RIN≥7.0: 0/1 (mediana 6.3)
PAXgene  + Quick     n= 12 | conc mediana    27.1 ng/µl | 260/280 mediana 0.85 | 260/230≥1.7: 0/12 | RIN≥7.0: 0/0 (mediana nan)
EDTA     + Quick     n=  8 | conc mediana   191.2 ng/µl | 260/280 media